# Bone Stress Prediction — Hybrid GCN-SAGE-Residual GNN

This notebook trains and evaluates the proposed hybrid GCN-SAGE-Residual graph neural
network for element-wise femoral stress prediction, as described in Section 2.7 of the
accompanying manuscript.

**Pipeline overview:**
1. Data is split into train/val/test **before** any normalization statistics are computed
   (prevents patient-level information leakage).
2. Stress normalization statistics (mean/std) are computed **from the training partition only**.
3. Test-set MAE/RMSE/R² are computed **after inverse-transforming predictions back to
   physical units (MPa)**.
4. Fracture/control group labels are attached to every graph for the subgroup analysis
   in Section 10.

See `../CHANGELOG.md` for a description of data-pipeline issues identified and corrected
during peer review (patient-level split verification, single-pass normalization).

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import glob
import re
import time
from tqdm import tqdm
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.cuda.amp import autocast, GradScaler
import gc

from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, SAGEConv, BatchNorm

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, precision_score, recall_score, f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("\n✓ All imports successful!")


## 2. Configuration

In [ ]:
# Update these paths to match your local data location (see data/README.md)
# GRAPH_DIR = r"./data/graphs"
# STRESS_DIR = r"./data/stress_labels"
# MODEL_DIR = "./models"
# os.makedirs(MODEL_DIR, exist_ok=True)

# Directory containing .pkl graph files
GRAPH_DIR = r"C:\Users\u232980\Bonestrength\Element_graph\1_16thJan\graphs_v2\training"

# Directory containing stress CSV files
STRESS_DIR = r"C:\Users\u232980\Bonestrength\Element_graph\7_Retraining_all_models\Node_Stress"

# Output directory for trained models
MODEL_DIR = "./models"
os.makedirs(MODEL_DIR, exist_ok=True)

STRESS_COLUMNS = ['S.S11', 'S.S22', 'S.S33', 'S.S12', 'S.S13', 'S.S23']

TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_STATE = 42

print("Configuration:")
print("=" * 60)
print(f"Graph directory:  {GRAPH_DIR}")
print(f"Stress directory: {STRESS_DIR}")
print(f"Model directory:  {MODEL_DIR}")
print(f"Stress columns:   {STRESS_COLUMNS}")


## 3. Find and Match Graph/Stress Files

In [ ]:
graph_files = sorted(glob.glob(os.path.join(GRAPH_DIR, "*_pyg.pkl")))
stress_files = sorted(glob.glob(os.path.join(STRESS_DIR, "*.csv")))

print(f"Found {len(graph_files)} graph files")
print(f"Found {len(stress_files)} stress files")


In [ ]:
def get_id_from_graph(filepath):
    """Extract numeric patient ID from graph filename."""
    basename = os.path.basename(filepath)
    name = basename.replace('_pyg.pkl', '').replace('_networkx.pkl', '')
    parts = name.split('_')
    for part in reversed(parts):
        if part.isdigit():
            return part
    match = re.search(r'(\d+)', name)
    return match.group(1) if match else None


def get_id_from_stress(filepath):
    """Extract numeric patient ID from stress-label filename."""
    basename = os.path.basename(filepath)
    name = os.path.splitext(basename)[0]
    if name.isdigit():
        return name
    parts = name.split('_')
    for part in reversed(parts):
        if part.isdigit():
            return part
    match = re.search(r'(\d+)', name)
    return match.group(1) if match else name


def get_group_from_graph(filepath):
    """
    Extract fracture/control label from filename prefix.
    Co_..._pyg.pkl -> 'control'
    Fx_..._pyg.pkl -> 'fracture'
    """
    basename = os.path.basename(filepath)
    if basename.startswith('Co_'):
        return 'control'
    elif basename.startswith('Fx_'):
        return 'fracture'
    return 'unknown'


print("\u2713 Helper functions defined")


In [ ]:
graph_dict = {get_id_from_graph(f): f for f in graph_files}
stress_dict = {get_id_from_stress(f): f for f in stress_files}

matched_pairs = []
for gid, gfile in graph_dict.items():
    if gid in stress_dict:
        matched_pairs.append({'id': gid, 'graph_file': gfile, 'stress_file': stress_dict[gid]})

unmatched_graphs = [gid for gid in graph_dict if gid not in stress_dict]
unmatched_stress = [sid for sid in stress_dict if sid not in graph_dict]

print("=" * 60)
print("MATCHING RESULTS")
print("=" * 60)
print(f"Matched pairs:    {len(matched_pairs)}")
print(f"Unmatched graphs: {len(unmatched_graphs)}")
print(f"Unmatched stress: {len(unmatched_stress)}")

prematch_groups = Counter(get_group_from_graph(p['graph_file']) for p in matched_pairs)
print(f"\nGroup balance (expect control:64, fracture:64): {dict(prematch_groups)}")


In [ ]:
if matched_pairs:
    sample_stress_file = matched_pairs[0]['stress_file']
    df = pd.read_csv(sample_stress_file)
    missing_cols = [c for c in STRESS_COLUMNS if c not in df.columns]
    if missing_cols:
        print(f"\u26a0 WARNING: Configured columns not found: {missing_cols}")
        print(f"  Available columns: {list(df.columns)}")
    else:
        print("\u2713 All stress columns found in sample file.")


## 4. Load Graphs — Raw (Unnormalized) Stress Targets

Stress values are kept in their original physical units (MPa) at load time.
Normalization is deferred until after the train/val/test split (Section 6).

In [ ]:
def load_graph_with_stress(graph_file, stress_file, stress_columns):
    """
    Load a graph and attach RAW (un-normalized) stress values as targets.
    Normalization must be fit only on the training partition, which does
    not exist yet at this point in the pipeline.
    """
    with open(graph_file, 'rb') as f:
        data = pickle.load(f)

    stress_df = pd.read_csv(stress_file)
    available_cols = [c for c in stress_columns if c in stress_df.columns]
    if len(available_cols) != len(stress_columns):
        missing = set(stress_columns) - set(available_cols)
        raise ValueError(f"Missing columns: {missing}. Available: {list(stress_df.columns)}")

    stress_values = stress_df[available_cols].values
    if len(stress_values) != data.num_nodes:
        raise ValueError(f"Node count mismatch: graph={data.num_nodes}, stress={len(stress_values)}")

    data.y = torch.tensor(stress_values, dtype=torch.float32)  # RAW, in MPa
    return data


print("\u2713 Data loading function defined (raw targets, no normalization yet)")


In [ ]:
all_data = []
failed = []

for pair in tqdm(matched_pairs, desc="Loading data"):
    try:
        data = load_graph_with_stress(pair['graph_file'], pair['stress_file'], STRESS_COLUMNS)
        data.name = pair['id']
        data.group = get_group_from_graph(pair['graph_file'])
        all_data.append(data)
    except Exception as e:
        failed.append({'id': pair['id'], 'error': str(e)})
        print(f"\u26a0 Failed to load {pair['id']}: {e}")

print(f"\u2713 Loaded {len(all_data)} graphs successfully")
if failed:
    print(f"\u2717 Failed to load {len(failed)} graphs")

# ---- Hard verification before proceeding ----
n_total = len(all_data)
n_unique = len(set(d.name for d in all_data))
group_totals = Counter(d.group for d in all_data)

print("\n" + "=" * 60)
print("PRE-SPLIT VERIFICATION")
print("=" * 60)
print(f"Total loaded:  {n_total}   (expect 128)")
print(f"Unique IDs:    {n_unique}  (expect 128)")
print(f"Group totals:  {dict(group_totals)}  (expect control:64, fracture:64)")

assert n_total == 128, f"Expected 128 graphs, got {n_total} -- check for duplicate loading!"
assert n_unique == 128, f"Expected 128 unique IDs, got {n_unique} -- duplicates present!"
assert group_totals.get('control', 0) == 64
assert group_totals.get('fracture', 0) == 64
print("\n\u2713 All checks passed.")

if all_data:
    sample = all_data[0]
    print(f"\nSample graph: ID={sample.name}, Group={sample.group}, "
          f"Nodes={sample.num_nodes:,}, Edges={sample.num_edges:,}, "
          f"Features={sample.x.shape}, Targets={sample.y.shape} (raw MPa)")


## 5. Split Data — Patient Level (Before Normalization)

The split is performed on raw data, before any normalization statistics are computed,
to prevent information leakage from validation/test patients into the normalization
used for training.

In [ ]:
EXPECTED_TEST_IDS = sorted(['0013900409', '0027027422', '0028174514', '0044034800',
    '0045633006', '0073786101', '0079307102', '0170816205', '0174810400', '0191844700',
    '0199546000', '0200190100', '0240547400', '1096308', '1587024', '1768405',
    '2085276', '244393', '298445', '986559'])

train_data, temp_data = train_test_split(all_data, test_size=(VAL_RATIO + TEST_RATIO), random_state=RANDOM_STATE)
val_ratio_adjusted = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
val_data, test_data = train_test_split(temp_data, test_size=(1 - val_ratio_adjusted), random_state=RANDOM_STATE)

print(f"Data split:")
print(f"  Train: {len(train_data)} graphs ({len(train_data)/len(all_data)*100:.1f}%)")
print(f"  Val:   {len(val_data)} graphs ({len(val_data)/len(all_data)*100:.1f}%)")
print(f"  Test:  {len(test_data)} graphs ({len(test_data)/len(all_data)*100:.1f}%)")

train_counts = Counter(d.group for d in train_data)
val_counts   = Counter(d.group for d in val_data)
test_counts  = Counter(d.group for d in test_data)
print("\nFracture/Control breakdown per split:")
print(f"  Train: {train_counts.get('fracture', 0)} fracture / {train_counts.get('control', 0)} control")
print(f"  Val:   {val_counts.get('fracture', 0)} fracture / {val_counts.get('control', 0)} control")
print(f"  Test:  {test_counts.get('fracture', 0)} fracture / {test_counts.get('control', 0)} control")

# Verify against the exact test-set patients reported in the manuscript. This guards
# against silent split changes caused by filesystem/glob ordering differences across machines.
test_ids_now = sorted([d.name for d in test_data])
assert test_ids_now == EXPECTED_TEST_IDS, (
    "Test set does not match the verified split reported in the manuscript. "
    "Check matched_pairs ordering before proceeding."
)
print("\n\u2713 Test set matches the verified split used for all reported results.")


## 6. Normalization — Fit on Training Partition Only

Mean/std are computed exclusively from `train_data` and applied unchanged to
validation and test data, preventing any leakage of held-out statistics into training.

In [ ]:
all_train_stress = torch.cat([d.y for d in train_data], dim=0)
train_stress_mean = all_train_stress.mean(dim=0)
train_stress_std = all_train_stress.std(dim=0) + 1e-8

print("Training-set stress statistics (MPa):")
print(f"{'Component':<15} {'Mean':>12} {'Std':>12}")
for i, col in enumerate(STRESS_COLUMNS):
    print(f"{col:<15} {train_stress_mean[i]:>12.4f} {train_stress_std[i]:>12.4f}")

for data in train_data + val_data + test_data:
    data.y = (data.y - train_stress_mean) / train_stress_std

global_stats = {
    'mean': train_stress_mean.numpy(),
    'std': train_stress_std.numpy(),
    'columns': STRESS_COLUMNS
}

with open(os.path.join(MODEL_DIR, 'stress_normalization_stats_TRAINONLY.pkl'), 'wb') as f:
    pickle.dump(global_stats, f)

print("\n\u2713 Normalized using training-only statistics (no leakage). Saved to disk.")


## 7. Hybrid GCN-SAGE-Residual Model

In [ ]:
class LargeHybridGNN(nn.Module):
    """
    Hybrid GCN-SAGE-Residual architecture (Section 2.7 of the manuscript).
    - 256 hidden channels, 6 layers total (3x GCN + 3x SAGE)
    - Residual connections bridging the GCN and SAGE stages
    - Deep output projection network
    """
    def __init__(self, in_channels, hidden_channels=256, out_channels=6, dropout=0.15):
        super().__init__()
        self.dropout = dropout

        self.gcn1 = GCNConv(in_channels, hidden_channels)
        self.bn1 = BatchNorm(hidden_channels)
        self.gcn2 = GCNConv(hidden_channels, hidden_channels)
        self.bn2 = BatchNorm(hidden_channels)
        self.gcn3 = GCNConv(hidden_channels, hidden_channels)
        self.bn3 = BatchNorm(hidden_channels)

        self.sage1 = SAGEConv(hidden_channels, hidden_channels)
        self.bn4 = BatchNorm(hidden_channels)
        self.sage2 = SAGEConv(hidden_channels, hidden_channels)
        self.bn5 = BatchNorm(hidden_channels)
        self.sage3 = SAGEConv(hidden_channels, hidden_channels)
        self.bn6 = BatchNorm(hidden_channels)

        self.residual1 = nn.Linear(in_channels, hidden_channels)
        self.residual2 = nn.Linear(hidden_channels, hidden_channels)

        self.output = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels), nn.BatchNorm1d(hidden_channels), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_channels, hidden_channels // 2), nn.BatchNorm1d(hidden_channels // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_channels // 2, hidden_channels // 4), nn.BatchNorm1d(hidden_channels // 4), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_channels // 4, out_channels)
        )

    def forward(self, x, edge_index, batch=None):
        x_input = x

        x = F.relu(self.bn1(self.gcn1(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x + self.residual1(x_input)

        x = F.relu(self.bn2(self.gcn2(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = F.relu(self.bn3(self.gcn3(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)

        x_res = x

        x = F.relu(self.bn4(self.sage1(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = F.relu(self.bn5(self.sage2(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = F.relu(self.bn6(self.sage3(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = x + self.residual2(x_res)
        return self.output(x)


class GentleWeightedMSELoss(nn.Module):
    """Quadratic-weighted MSE emphasizing high-stress nodes (Section 2.7.1)."""
    def __init__(self, high_stress_weight=2.0):
        super().__init__()
        self.high_stress_weight = high_stress_weight

    def forward(self, pred, target):
        target_magnitude = torch.sqrt((target ** 2).sum(dim=1))
        percentile = target_magnitude / (target_magnitude.max() + 1e-8)
        weights = 1.0 + (self.high_stress_weight - 1.0) * (percentile ** 2)
        mse = ((pred - target) ** 2).mean(dim=1)
        return (mse * weights).mean()


print("\u2713 LargeHybridGNN and GentleWeightedMSELoss defined")


In [ ]:
torch.cuda.empty_cache()
gc.collect()

CONFIG = {
    'hidden_channels': 256,
    'dropout': 0.15,
    'epochs': 10,
    'batch_size': 4,
    'learning_rate': 0.0005,
    'weight_decay': 1e-6,
    'min_lr': 0.00001,
    'scheduler_T0': 300,
    'scheduler_Tmult': 2,
    'patience': 50,
    'high_stress_weight': 2.0,
    'model_dir': MODEL_DIR,
    'model_name': 'LargeHybridGNN'
}

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

model = LargeHybridGNN(
    in_channels=train_data[0].x.shape[1],
    hidden_channels=CONFIG['hidden_channels'],
    out_channels=train_data[0].y.shape[1],
    dropout=CONFIG['dropout']
).to(device)
print(f"\n\u2713 Model: {sum(p.numel() for p in model.parameters()):,} parameters")

train_loader = DataLoader(train_data, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_data, batch_size=CONFIG['batch_size'], shuffle=False)
test_loader = DataLoader(test_data, batch_size=CONFIG['batch_size'], shuffle=False)
print(f"\u2713 Loaders ready (batch_size={CONFIG['batch_size']})")

criterion = GentleWeightedMSELoss(CONFIG['high_stress_weight'])
print("\u2713 Loss function ready")


## 8. Training Setup

In [ ]:
optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=CONFIG['scheduler_T0'], T_mult=CONFIG['scheduler_Tmult'], eta_min=CONFIG['min_lr'])
scaler = GradScaler()


def train_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        with autocast():
            out = model(batch.x, batch.edge_index)
            loss = criterion(out, batch.y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * batch.num_graphs
        del out, loss, batch
        torch.cuda.empty_cache()
    return total_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device):
    # Returns loss and NORMALIZED predictions/targets.
    model.eval()
    total_loss = 0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index)
            loss = criterion(out, batch.y)
            total_loss += loss.item() * batch.num_graphs
            all_preds.append(out.cpu())
            all_targets.append(batch.y.cpu())
            del out, loss, batch
            torch.cuda.empty_cache()
    return total_loss / len(loader.dataset), torch.cat(all_preds), torch.cat(all_targets)


print("\u2713 Training and evaluation functions ready")


## 9. Training Loop

In [ ]:
print("=" * 70)
print("TRAINING")
print("=" * 70)

best_r2 = 0
patience_counter = 0
start_time = time.time()

for epoch in range(CONFIG['epochs']):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, scaler, device)
    val_loss, predictions, targets = evaluate(model, val_loader, criterion, device)
    val_r2 = r2_score(targets[:, 0].numpy(), predictions[:, 0].numpy())
    scheduler.step()

    improved = val_r2 > best_r2
    print(f"Epoch {epoch+1:3d}/{CONFIG['epochs']} | Train: {train_loss:.6f} | "
          f"Val: {val_loss:.6f} | R\u00b2: {val_r2:.4f}{' *' if improved else ''}")

    if improved:
        best_r2 = val_r2
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_r2': val_r2,
            'config': CONFIG,
            'global_stats': global_stats
        }, os.path.join(CONFIG['model_dir'], f"best_model_{CONFIG['model_name']}.pt"))
    else:
        patience_counter += 1
        if patience_counter >= CONFIG['patience']:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

    if (epoch + 1) % 10 == 0:
        gc.collect()

print(f"\nBest validation R\u00b2 (normalized): {best_r2:.4f} | "
      f"Training time: {(time.time()-start_time)/3600:.2f}h")


## 10. Test Set Evaluation — Metrics in Physical Units (MPa)

Predictions and targets are inverse-transformed back to MPa using the training-only
statistics saved with the checkpoint, before computing MAE/RMSE/R².

In [ ]:
def compute_metrics_mpa(preds_norm, targets_norm, stress_columns, mean, std):
    """Inverse-transform to MPa BEFORE computing any metric."""
    preds_norm = preds_norm.numpy() if isinstance(preds_norm, torch.Tensor) else preds_norm
    targets_norm = targets_norm.numpy() if isinstance(targets_norm, torch.Tensor) else targets_norm

    preds_mpa = preds_norm * std + mean
    targets_mpa = targets_norm * std + mean

    metrics = {'overall': {
        'mse': mean_squared_error(targets_mpa, preds_mpa),
        'rmse': np.sqrt(mean_squared_error(targets_mpa, preds_mpa)),
        'mae': mean_absolute_error(targets_mpa, preds_mpa),
        'r2': r2_score(targets_mpa, preds_mpa)
    }}
    for i, col in enumerate(stress_columns[:preds_mpa.shape[1]]):
        metrics[col] = {
            'mae': mean_absolute_error(targets_mpa[:, i], preds_mpa[:, i]),
            'rmse': np.sqrt(mean_squared_error(targets_mpa[:, i], preds_mpa[:, i])),
            'r2': r2_score(targets_mpa[:, i], preds_mpa[:, i])
        }
    return metrics, preds_mpa, targets_mpa


# Load the best checkpoint (works even if this cell is re-run in a fresh session)
best_ckpt_path = os.path.join(CONFIG['model_dir'], f"best_model_{CONFIG['model_name']}.pt")
checkpoint = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
ckpt_stats = checkpoint['global_stats']
model.eval()

test_loss, test_preds_norm, test_targets_norm = evaluate(model, test_loader, criterion, device)
metrics_mpa, preds_mpa, targets_mpa = compute_metrics_mpa(
    test_preds_norm, test_targets_norm, STRESS_COLUMNS, ckpt_stats['mean'], ckpt_stats['std']
)

r2_label = 'R\u00b2'
print("=" * 70)
print("  TEST SET METRICS -- PHYSICAL UNITS (MPa)")
print("=" * 70)
print(f"  MSE:  {metrics_mpa['overall']['mse']:.6f}")
print(f"  RMSE: {metrics_mpa['overall']['rmse']:.6f} MPa")
print(f"  MAE:  {metrics_mpa['overall']['mae']:.6f} MPa")
print(f"  {r2_label}:   {metrics_mpa['overall']['r2']:.4f}")

print(f"\nPer-Component Performance:")
print(f"  {'Component':<12} {r2_label:>8} {'RMSE (MPa)':>12} {'MAE (MPa)':>12}")
for col in STRESS_COLUMNS:
    print(f"  {col:<12} {metrics_mpa[col]['r2']:>8.4f} {metrics_mpa[col]['rmse']:>12.6f} {metrics_mpa[col]['mae']:>12.6f}")


## 11. Per-Component Prediction Plots (Figure 5)

In [ ]:
STRESS_LABELS = ['S11', 'S22', 'S33', 'S12', 'S13', 'S23']

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

overall_r2 = metrics_mpa['overall']['r2']
fig.suptitle(f"Test Set Predictions: Hybrid GCN-SAGE-Residual\nOverall R\u00b2 = {overall_r2:.4f}",
             fontsize=15, fontweight='bold')

n_plot = min(20000, targets_mpa.shape[0])
plot_idx = np.random.choice(targets_mpa.shape[0], n_plot, replace=False)

for i, (ax, label) in enumerate(zip(axes, STRESS_LABELS)):
    y_true, y_pred = targets_mpa[plot_idx, i], preds_mpa[plot_idx, i]
    r2_i = metrics_mpa[STRESS_COLUMNS[i]]['r2']
    rmse_i = metrics_mpa[STRESS_COLUMNS[i]]['rmse']

    ax.scatter(y_true, y_pred, s=3, alpha=0.25, color='tab:blue', edgecolors='none')
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect')
    coeffs = np.polyfit(y_true, y_pred, 1)
    ax.plot(lims, np.poly1d(coeffs)(lims), 'g-', linewidth=1.5, label=f'Fit (y={coeffs[0]:.2f}x+{coeffs[1]:.2f})')

    ax.set_title(f"S.{label}\nR\u00b2 = {r2_i:.4f}, RMSE = {rmse_i:.4f}", fontsize=12)
    ax.set_xlabel("Actual Stress (MPa)")
    ax.set_ylabel("Predicted Stress (MPa)")
    ax.legend(fontsize=8, loc='upper left')
    ax.set_xlim(lims)
    ax.set_ylim(lims)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(os.path.join(MODEL_DIR, 'figure5_per_component.png'), dpi=300, bbox_inches='tight')
plt.show()


## 12. Hotspot Detection (Section 3.4.1)

Hotspots are defined as the top 10% of elements by von Mises equivalent stress,
thresholded independently per patient for both ground-truth and predicted fields.

In [ ]:
def von_mises_stress(stress_tensor_6):
    """Von Mises equivalent stress from the 6 independent Cauchy components."""
    s11, s22, s33, s12, s13, s23 = [stress_tensor_6[:, i] for i in range(6)]
    return np.sqrt(0.5 * ((s11 - s22)**2 + (s22 - s33)**2 + (s33 - s11)**2
                           + 6 * (s12**2 + s13**2 + s23**2)))


def compute_hotspot_metrics(model, data_list, device, mean, std, percentile=90):
    """
    For each patient: compute von Mises stress (actual + predicted), threshold at
    the patient's own percentile, then report pooled (across all elements) and
    per-patient macro-averaged precision/recall/F1.
    """
    model.eval()
    all_actual_mask, all_pred_mask, per_patient_results = [], [], []

    with torch.no_grad():
        for data in data_list:
            data_gpu = data.to(device)
            pred_norm = model(data_gpu.x, data_gpu.edge_index).cpu().numpy()
            target_norm = data.y.cpu().numpy()

            pred_mpa = pred_norm * std + mean
            target_mpa = target_norm * std + mean

            vm_actual = von_mises_stress(target_mpa)
            vm_pred = von_mises_stress(pred_mpa)

            thresh_actual = np.percentile(vm_actual, percentile)
            thresh_pred = np.percentile(vm_pred, percentile)
            actual_mask = (vm_actual > thresh_actual).astype(int)
            pred_mask = (vm_pred > thresh_pred).astype(int)

            all_actual_mask.append(actual_mask)
            all_pred_mask.append(pred_mask)
            per_patient_results.append({
                'patient': data.name,
                'precision': precision_score(actual_mask, pred_mask, zero_division=0),
                'recall': recall_score(actual_mask, pred_mask, zero_division=0),
                'f1': f1_score(actual_mask, pred_mask, zero_division=0)
            })

    pooled_actual = np.concatenate(all_actual_mask)
    pooled_pred = np.concatenate(all_pred_mask)
    pooled = {
        'precision': precision_score(pooled_actual, pooled_pred, zero_division=0),
        'recall': recall_score(pooled_actual, pooled_pred, zero_division=0),
        'f1': f1_score(pooled_actual, pooled_pred, zero_division=0)
    }
    macro = {
        'precision': np.mean([r['precision'] for r in per_patient_results]),
        'recall': np.mean([r['recall'] for r in per_patient_results]),
        'f1': np.mean([r['f1'] for r in per_patient_results])
    }
    return {'pooled': pooled, 'macro': macro, 'per_patient': per_patient_results}


hotspot_results = compute_hotspot_metrics(model, test_data, device, ckpt_stats['mean'], ckpt_stats['std'], percentile=90)

print("=" * 70)
print("  HOTSPOT DETECTION -- top 10% von Mises stress elements")
print("=" * 70)
print(f"\nPooled (all test-set elements):")
print(f"  Precision = {hotspot_results['pooled']['precision']:.4f}")
print(f"  Recall    = {hotspot_results['pooled']['recall']:.4f}")
print(f"  F1        = {hotspot_results['pooled']['f1']:.4f}")
print(f"\nPer-patient macro-average (n={len(test_data)} test patients):")
print(f"  Precision = {hotspot_results['macro']['precision']:.4f}")
print(f"  Recall    = {hotspot_results['macro']['recall']:.4f}")
print(f"  F1        = {hotspot_results['macro']['f1']:.4f}")


## 13. Fracture vs. Control Subgroup Performance (Section 3.4.2)

In [ ]:
def evaluate_subgroup(model, data_list, group_name, device, mean, std, stress_columns):
    subset = [d for d in data_list if d.group == group_name]
    if not subset:
        print(f"No test graphs found for group '{group_name}'")
        return None, 0
    loader = DataLoader(subset, batch_size=CONFIG['batch_size'], shuffle=False)
    _, preds_norm, targets_norm = evaluate(model, loader, criterion, device)
    metrics, _, _ = compute_metrics_mpa(preds_norm, targets_norm, stress_columns, mean, std)
    return metrics, len(subset)


fx_metrics, fx_n = evaluate_subgroup(model, test_data, 'fracture', device, ckpt_stats['mean'], ckpt_stats['std'], STRESS_COLUMNS)
co_metrics, co_n = evaluate_subgroup(model, test_data, 'control', device, ckpt_stats['mean'], ckpt_stats['std'], STRESS_COLUMNS)

print("Fracture vs. Control test-set performance (MPa):\n")
if fx_metrics:
    print(f"Fracture subset (n={fx_n} patients):")
    print(f"  {r2_label} = {fx_metrics['overall']['r2']:.4f}, MAE = {fx_metrics['overall']['mae']:.4f} MPa, RMSE = {fx_metrics['overall']['rmse']:.4f} MPa")
if co_metrics:
    print(f"\nControl subset (n={co_n} patients):")
    print(f"  {r2_label} = {co_metrics['overall']['r2']:.4f}, MAE = {co_metrics['overall']['mae']:.4f} MPa, RMSE = {co_metrics['overall']['rmse']:.4f} MPa")


## 14. Computational Efficiency Benchmark (Section 4, Table 6)

Per-patient inference timing, following GPU warm-up to exclude one-time CUDA
initialization overhead. Reports the speed-up relative to the ~51-minute FE simulation.

In [ ]:
import platform
import psutil

print("=" * 60)
print("HARDWARE CONFIGURATION")
print("=" * 60)
print(f"CPU: {platform.processor()}")
print(f"CPU cores: {psutil.cpu_count(logical=False)} physical / {psutil.cpu_count(logical=True)} logical")
print(f"System RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

n_bench = min(10, len(test_data))
bench_patients = test_data[:n_bench]

# Data preprocessing time (load raw graph + stress CSV from disk)
preprocessing_times = []
for data in bench_patients:
    stress_file = next(p['stress_file'] for p in matched_pairs if p['id'] == data.name)
    graph_file = next(p['graph_file'] for p in matched_pairs if p['id'] == data.name)
    t0 = time.perf_counter()
    with open(graph_file, 'rb') as f:
        _ = pickle.load(f)
    _ = pd.read_csv(stress_file)
    preprocessing_times.append(time.perf_counter() - t0)
preprocessing_times = np.array(preprocessing_times)

# Thorough GPU warm-up: run the full benchmark set once, discarded, before timing
model.eval()
with torch.no_grad():
    for data in bench_patients:
        data_gpu = data.to(device)
        _ = model(data_gpu.x, data_gpu.edge_index)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

# Timed pass: device transfer + steady-state inference
inference_times, data_transfer_times = [], []
with torch.no_grad():
    for data in bench_patients:
        t0 = time.perf_counter()
        data_gpu = data.to(device)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        _ = model(data_gpu.x, data_gpu.edge_index)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t2 = time.perf_counter()

        data_transfer_times.append(t1 - t0)
        inference_times.append(t2 - t1)

inference_times = np.array(inference_times)
data_transfer_times = np.array(data_transfer_times)

print(f"\n{'=' * 60}")
print("PER-PATIENT TIMING (steady-state, after warm-up)")
print("=" * 60)
print(f"Data preprocessing: {preprocessing_times.mean()*1000:.2f} \u00b1 {preprocessing_times.std()*1000:.2f} ms")
print(f"Device transfer:    {data_transfer_times.mean()*1000:.2f} \u00b1 {data_transfer_times.std()*1000:.2f} ms")
print(f"Model inference:    {inference_times.mean()*1000:.2f} \u00b1 {inference_times.std()*1000:.2f} ms")

total_time = preprocessing_times.mean() + data_transfer_times.mean() + inference_times.mean()
print(f"\nTotal GNN surrogate time: {total_time*1000:.2f} ms")

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        data_gpu = bench_patients[0].to(device)
        _ = model(data_gpu.x, data_gpu.edge_index)
        torch.cuda.synchronize()
    print(f"Peak GPU memory (single-patient inference): {torch.cuda.max_memory_allocated()/1e9:.3f} GB")

FE_TIME_SECONDS = 51 * 60
print(f"\n{'=' * 60}")
print("SPEED-UP RATIO")
print("=" * 60)
print(f"FE simulation time:  {FE_TIME_SECONDS/60:.1f} min")
print(f"GNN total time:      {total_time:.4f} s")
print(f"Speed-up factor:     {FE_TIME_SECONDS/total_time:,.0f}x")


## 15. Software and Hardware Versions (Reproducibility)

In [ ]:
import torch_geometric
import sklearn
import sys

print("=" * 60)
print("SOFTWARE / LIBRARY VERSIONS")
print("=" * 60)
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA (via PyTorch): {torch.version.cuda}")
print(f"cuDNN: {torch.backends.cudnn.version()}")
print(f"PyTorch Geometric: {torch_geometric.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"\nOS: {platform.platform()}")


# 16.  Fracture vs. Control Subgroup Performance

In [ ]:
def get_group_from_id(patient_id, matched_pairs):
    entry = next(p for p in matched_pairs if p['id'] == patient_id)
    basename = os.path.basename(entry['graph_file'])
    if basename.startswith('Co_'):
        return 'control'
    elif basename.startswith('Fx_'):
        return 'fracture'
    return 'unknown'

for d in test_data:
    d.group = get_group_from_id(d.name, matched_pairs)

print("\u2713 .group attached to test_data")
print(Counter(d.group for d in test_data))  # sanity check: expect {'fracture': 12, 'control': 8}

In [ ]:
def evaluate_subgroup(model, data_list, group_name, device, mean, std, stress_columns):
    subset = [d for d in data_list if d.group == group_name]
    if not subset:
        print(f"No test graphs found for group '{group_name}'")
        return None, 0
    loader = DataLoader(subset, batch_size=CONFIG['batch_size'], shuffle=False)
    _, preds_norm, targets_norm = evaluate(model, loader, criterion, device)
    metrics, _, _ = compute_metrics_mpa(preds_norm, targets_norm, stress_columns, mean, std)
    return metrics, len(subset)

fx_metrics, fx_n = evaluate_subgroup(model, test_data, 'fracture', device, ckpt_stats['mean'], ckpt_stats['std'], STRESS_COLUMNS)
co_metrics, co_n = evaluate_subgroup(model, test_data, 'control', device, ckpt_stats['mean'], ckpt_stats['std'], STRESS_COLUMNS)

In [ ]:

r2_label = 'R\u00b2'
if fx_metrics:
    print(f"Fracture subset (n={fx_n} patients):")
    print(f"  {r2_label} = {fx_metrics['overall']['r2']:.4f}, MAE = {fx_metrics['overall']['mae']:.4f} MPa, RMSE = {fx_metrics['overall']['rmse']:.4f} MPa")
if co_metrics:
    print(f"\nControl subset (n={co_n} patients):")
    print(f"  {r2_label} = {co_metrics['overall']['r2']:.4f}, MAE = {co_metrics['overall']['mae']:.4f} MPa, RMSE = {co_metrics['overall']['rmse']:.4f} MPa")